# EDA - Movie Recommendation System

Quick exploratory look at the MovieLens raw data and the processed metadata table.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_all, quick_summary
from src.data.cleaning import build_processed_table

## 1. Load raw data

In [ ]:
dataset = load_all(raw_dir='../data/raw')
for k, v in quick_summary(dataset).items():
    print(f'{k}: {v}')

## 2. Missing values check

In [ ]:
for name, df in dataset.items():
    print(name, '->', df.isna().sum().sum(), 'missing values')

## 3. Rating distribution

In [ ]:
dataset['ratings']['rating'].value_counts().sort_index().plot(kind='bar', figsize=(8,4), title='Rating distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

## 4. Genre frequency

In [ ]:
genre_counts = dataset['movies']['genres'].str.split('|').explode().value_counts()
genre_counts.head(15).plot(kind='barh', figsize=(8,6), title='Top genres')
plt.gca().invert_yaxis()
plt.show()

## 5. Build processed metadata and preview

In [ ]:
processed = build_processed_table(dataset)
processed[['movieId','clean_title','release_year','metadata_text']].head(10)

## 6. Quick recommendation sanity check

In [ ]:
from src.features.vectorize import build_features
from src.recommender.engine import RecommenderEngine

bundle = build_features(processed)
engine = RecommenderEngine(bundle.movies_df, bundle.tfidf_matrix, bundle.id_to_row, bundle.row_to_id)

for r in engine.recommend_by_title('Toy Story', top_n=5, min_rating_count=10):
    print(r['title'], '->', r['similarity_score'])